In [ ]:
using LinearAlgebra

# Forward-mode AD from scratch

*Adapted from:* https://michael-herbst.com/talks/2025.10.09_IPAM_DFT_Gradients_1_ad.html

We will provide a simplistic, but functional forward-mode AD implementation. The basic idea is to move synchronously "left-to-right" in the flowchart, that is to compute the value and derivatives synchronously.

We define a new number type for doing that:

In [ ]:
struct Dual <: Number
    x::Float64   # Value
    δx::Float64  # Derivative
end

This number type we now equip with the basic differentiation rules:

In [ ]:
begin
	# (f+g)'(x) = f'(x) + g'(x) 
	Base.:+(a::Dual, b::Dual) = Dual(a.x + b.x, a.δx + b.δx)
	Base.:-(a::Dual, b::Dual) = Dual(a.x - b.x, a.δx - b.δx)

	# (f*g)'(x) = f(x)*g'(x) + f'(x)*g(x)
	Base.:*(a::Dual, b::Dual) = Dual(a.x * b.x, a.x * b.δx + a.δx * b.x )
	Base.:/(a::Dual, b::Dual) = Dual(a.x / b.x, (b.x * a.δx - a.x  * b.δx) / b.x^2)
end

On top of this we need to tell julia, that any number is also a dual number, just with an empty derivative:



In [ ]:
begin
	Base.convert(::Type{Dual}, x::Real) = Dual(x, zero(x))
	Base.promote_rule(::Type{Dual}, ::Type{<:Number}) = Dual
end

A derivative is now obtained by starting with a unit derivative and the desired value and just propagating through. We try to compute
```math
\left. \frac{d}{dx} \left(  x^3 - x^2 \right) \right|_{x=2} = 3 \cdot 2^2 - 2\cdot2 = 8
```
We introduce:

In [ ]:
derivative(f, x::Number) = f(  Dual(x, one(x))  )

and compute:

In [ ]:
derivative(x -> x^3 - x^2, 2.0)